In [ ]:
import os
import glob
import torch
from tqdm import tqdm

def precise_check_pyg_data_health(data_dir):
    search_pattern = os.path.join(data_dir, '**', '*.pt')
    file_list = sorted(glob.glob(search_pattern, recursive=True))
    
    if not file_list:
        print(f"未在 {data_dir} 找到任何 .pt 文件。")
        return

    print(f"开始深度扫描 {len(file_list)} 个 PyG 数据文件...")
    
    # =========================================================================
    # 重新定义的严谨排查阈值 (基于 MEB 中心和 Z-score 归一化)
    # =========================================================================
    # 1. 坐标阈值: 中心已归零，口袋截断为 12Å。考虑到边缘碎片质心可能略微超出，
    # 设定 18.0Å 为物理空间硬边界。
    MAX_POS_THRESHOLD = 24.0
    MAX_REF_THRESHOLD = 24.0 

    # 2. 化学属性阈值: 经过 Z-score 归一化后，绝大部分值应在 [-3, 3] 区间。
    # 设定 15.0 (即偏离均值 15 倍标准差) 作为捕捉极端异常值的绝对阈值。
    MAX_CHEM_THRESHOLD = 6
    
    # 3. 嵌入向量阈值: ChemBERTa 投影后的合理范围
    MAX_EMBED_THRESHOLD = 50.0

    EXPECTED_EMBED_DIM = 768

    # 异常分类统计
    issues = {
        'Empty_Graph_Zero_Nodes': [], 
        'Embedding_Dim_Mismatch': [],
        
        'NaN_or_Inf_in_Embeddings': [],
        'NaN_or_Inf_in_ChemProps': [],
        'NaN_or_Inf_in_Pos': [],
        'NaN_or_Inf_in_RefCoords': [],
        
        'Z_Score_Anomaly_ChemProps': [], # 归一化后的超级离群值
        'Extreme_Embeddings': [],
        'Out_of_Bounds_Pos': [],
        'Out_of_Bounds_Ref': [],
        
        'Zero_Ref_Collapsed': []
    }

    for f_path in tqdm(file_list, desc="Scanning", mininterval=1.0):
        try:
            data = torch.load(f_path, map_location='cpu', weights_only=False)
        except Exception as e:
            continue
            
        ligand_id = data.ligand_id if hasattr(data, 'ligand_id') else os.path.basename(f_path)
        
        # --- 0. 检查空图 ---
        if not hasattr(data, 'x') or data.num_nodes == 0 or data.x.shape[0] == 0:
            issues['Empty_Graph_Zero_Nodes'].append(ligand_id)
            continue 
        
        # --- 1. 精确拆解图特征 ---
        # 提取 14 维化学特征 (跳过 data.x 前2维的 node_type [1.0, 0.0])
        if data.x.shape[1] >= 16:
            chem_props = data.x[:, 2:] 
        else:
            chem_props = torch.empty(0)
            
        # 提取语言模型嵌入向量
        embeds = data.frag_embeds if hasattr(data, 'frag_embeds') else torch.empty(0)

        # --- 2. 深度数值体检 ---
        
        # 检查 ChemProps (Z-score 异常检测)
        if chem_props.numel() > 0:
            if torch.isnan(chem_props).any() or torch.isinf(chem_props).any():
                issues['NaN_or_Inf_in_ChemProps'].append(ligand_id)
            else:
                chem_max = chem_props.abs().max().item()
                if chem_max > MAX_CHEM_THRESHOLD:
                    issues['Z_Score_Anomaly_ChemProps'].append((ligand_id, chem_max))
                    
        # 检查 Embeddings
        if embeds.numel() > 0:
            # 假设 embeds 的形状是 [num_nodes, embed_dim]
            if len(embeds.shape) != 2 or embeds.shape[1] != EXPECTED_EMBED_DIM:
                issues['Embedding_Dim_Mismatch'].append((ligand_id, list(embeds.shape)))
                
            if torch.isnan(embeds).any() or torch.isinf(embeds).any():
                issues['NaN_or_Inf_in_Embeddings'].append(ligand_id)
            else:
                embed_max = embeds.abs().max().item()
                if embed_max > MAX_EMBED_THRESHOLD:
                    issues['Extreme_Embeddings'].append((ligand_id, embed_max))
            
        # 检查 3D 坐标 data.pos
        if hasattr(data, 'pos') and data.pos.numel() > 0:
            if torch.isnan(data.pos).any() or torch.isinf(data.pos).any():
                issues['NaN_or_Inf_in_Pos'].append(ligand_id)
            else:
                max_dist_to_origin = torch.norm(data.pos, dim=1).max().item()
                if max_dist_to_origin > MAX_POS_THRESHOLD:
                    issues['Out_of_Bounds_Pos'].append((ligand_id, max_dist_to_origin))
                    
        # 检查参考框架 data.ref_coords
        if hasattr(data, 'ref_coords') and data.ref_coords.numel() > 0:
            if torch.isnan(data.ref_coords).any() or torch.isinf(data.ref_coords).any():
                issues['NaN_or_Inf_in_RefCoords'].append(ligand_id)
            else:
                max_ref_len = data.ref_coords.abs().max().item()
                if max_ref_len > MAX_REF_THRESHOLD:
                    issues['Out_of_Bounds_Ref'].append((ligand_id, max_ref_len))
                
                norms = torch.norm(data.ref_coords.view(data.ref_coords.shape[0], -1), dim=1)
                if (norms == 0.0).any():
                    issues['Zero_Ref_Collapsed'].append(ligand_id)

    # --- 3. 打印深度诊断报告 ---
    print("\n" + "="*65)
    print("�� PyG 数据深度健康检查报告 (基于 MEB 归零与 Z-score 排查)")
    print("="*65)
    
    total_issues = 0
    for issue_name, affected_files in issues.items():
        count = len(affected_files)
        total_issues += count
        if count > 0:
            print(f"⚠️ [{issue_name}]: 发现 {count} 个异常!")
            if "Extreme" in issue_name or "Anomaly" in issue_name or "Out_of_Bounds" in issue_name:
                affected_files.sort(key=lambda x: x[1], reverse=True)
                sample_list = affected_files[:5]
                print(f"   -> 严重示例 (ID, 异常峰值):")
                for item in sample_list:
                    print(f"      - {item[0]}: {item[1]:.2f}")
            else:
                sample_list = affected_files[:5]
                print(f"   -> 示例 ID: {sample_list}")
            print("-" * 65)
            
    if total_issues == 0:
        print("✅ 完美！数据干干净净，无任何数值异常。")
    else:
        print(f"❌ 总计发现 {total_issues} 处异常。")

if __name__ == '__main__':
    TARGET_DIR = "/home/yyy/miniconda3/envs/PhiStone/project/GNN/CHEMCUT/processed_data/SE3TD_Pre_datas_768"
    precise_check_pyg_data_health(TARGET_DIR)

In [ ]:
# =============================================================================
# 专项清理: 物理删除存在 Zero_Ref_Collapsed (参考系崩塌) 的劣质数据
# =============================================================================
import os
import glob
import torch
from tqdm.auto import tqdm

def delete_collapsed_ref_files(data_dir):
    search_pattern = os.path.join(data_dir, '**', '*.pt')
    file_list = sorted(glob.glob(search_pattern, recursive=True))
    
    if not file_list:
        print(f"未在 {data_dir} 找到任何 .pt 文件。")
        return

    print(f"开始扫描并清理 {len(file_list)} 个 PyG 数据文件...")
    
    deleted_count = 0
    error_count = 0
    
    for f_path in tqdm(file_list, desc="Cleaning Collapsed Refs"):
        try:
            # 只需要读取张量结构，不需要加载图到 GPU
            data = torch.load(f_path, map_location='cpu', weights_only=False)
        except Exception:
            error_count += 1
            continue
            
        is_collapsed = False
        
        # 检查参考框架是否折叠成了一个点 (范数为0)
        if hasattr(data, 'ref_coords') and data.ref_coords.numel() > 0:
            # 展平后计算 L2 范数
            norms = torch.norm(data.ref_coords.view(data.ref_coords.shape[0], -1), dim=1)
            if (norms == 0.0).any():
                is_collapsed = True
                
        # 发现崩塌，执行物理删除
        if is_collapsed:
            try:
                os.remove(f_path)
                deleted_count += 1
            except Exception as e:
                print(f"\n无法删除文件 {f_path}: {e}")
                
    print("\n" + "="*60)
    print("�� 清理行动完成！")
    print(f"✅ 成功物理删除了 {deleted_count} 个存在 Zero_Ref_Collapsed 异常的 .pt 文件。")
    if error_count > 0:
        print(f"⚠️ 另有 {error_count} 个文件无法正常加载读取。")
    print("="*60)

if __name__ == '__main__':
    # 指向你刚刚生成好的 768 维度的数据目录
    TARGET_DIR = "/home/yyy/miniconda3/envs/PhiStone/project/GNN/CHEMCUT/processed_data/SE3TD_fine_datas_768"
    delete_collapsed_ref_files(TARGET_DIR)

In [ ]:
import os
import glob
import torch
from collections import Counter
from tqdm import tqdm

def deep_dive_chem_anomalies(data_dir, max_display=20):
    search_pattern = os.path.join(data_dir, '**', '*.pt')
    file_list = sorted(glob.glob(search_pattern, recursive=True))
    
    if not file_list:
        print(f"未在 {data_dir} 找到任何 .pt 文件。")
        return

    MAX_CHEM_THRESHOLD = 6.0
    
    anomalies_list = []
    dim_fail_counter = Counter()

    print(f"开始深度扫描，定位具体是哪几维化学特征出现异常...")
    
    for f_path in tqdm(file_list, desc="Scanning ChemProps"):
        try:
            data = torch.load(f_path, map_location='cpu', weights_only=False)
        except Exception:
            continue
            
        # 提取 14 维连续化学特征 (跳过前两维的节点类型 one-hot)
        if not hasattr(data, 'x') or data.x.shape[1] < 16:
            continue
            
        ligand_id = data.ligand_id if hasattr(data, 'ligand_id') else os.path.basename(f_path)
        chem_props = data.x[:, 2:] 
        
        if chem_props.numel() == 0 or torch.isnan(chem_props).any() or torch.isinf(chem_props).any():
            continue
        
        # 寻找每个特征维度的绝对值的最大值及其所在节点的索引
        # shape: [14]
        max_abs_vals, max_abs_indices = chem_props.abs().max(dim=0)
        
        # 找出绝对值超过设定阈值 (6.0) 的维度索引
        outlier_dims = torch.where(max_abs_vals > MAX_CHEM_THRESHOLD)[0].tolist()
        
        if outlier_dims:
            details = {}
            for dim in outlier_dims:
                dim_fail_counter[dim] += 1
                # 根据索引，取出该特征维度上实际带有符号的原始极端值
                node_idx = max_abs_indices[dim].item()
                actual_val = chem_props[node_idx, dim].item()
                details[dim] = actual_val
                
            anomalies_list.append({
                'ligand_id': ligand_id,
                'details': details
            })

    # ==========================================
    # 打印诊断报告
    # ==========================================
    print("\n" + "="*70)
    print(f"�� 化学特征 (ChemProps) 异常维度深度诊断报告")
    print("="*70)
    print(f"总计发现 {len(anomalies_list)} 个化合物存在化学特征越界 (绝对值 > {MAX_CHEM_THRESHOLD})。")
    
    if not anomalies_list:
        print("✅ 恭喜！没有发现任何超过阈值的化学特征。")
        return
        
    print("\n�� 异常维度全局统计 (哪个特征最容易出问题？):")
    # 按照出错次数降序排列
    for dim, count in dim_fail_counter.most_common():
        print(f"  - 特征维度 {dim} (对应 data.x 第 {dim+2} 列) : 共出现 {count} 次越界")
        
    print(f"\n�� 前 {max_display} 个异常化合物详细名单:")
    # 按照异常维度的严重程度或简单取前 max_display 个展示
    for i, item in enumerate(anomalies_list[:max_display]):
        print(f"[{i+1}] 化合物 ID: {item['ligand_id']}")
        for dim, val in item['details'].items():
            print(f"    ↳ 维度 {dim} 异常: 极端值为 {val:.3f}")
            
    print("="*70)

if __name__ == '__main__':
    TARGET_DIR = "/home/yyy/miniconda3/envs/PhiStone/project/GNN/CHEMCUT/processed_data/SE3TD_fine_datas_768"
    deep_dive_chem_anomalies(TARGET_DIR, max_display=20)